In [ ]:
# v15 FULL PIPELINEimport pandas as pdimport numpy as npfrom sklearn.model_selection import StratifiedKFoldfrom lightgbm import LGBMClassifierfrom sklearn.metrics import roc_auc_scoretrain = pd.read_csv("train.csv")test = pd.read_csv("test.csv")target = "임신 성공 여부"def preprocess(df):    df = df.copy()    if "시술 당시 나이" in df.columns:        df["age_num"] = df["시술 당시 나이"].str.extract(r'(\d+)').astype(float)    for col in df.columns:        if df[col].isnull().sum() > 0:            df[col + "_isnull"] = df[col].isnull().astype(int)    return dftrain = preprocess(train)test = preprocess(test)features = [c for c in train.columns if c not in [target, "ID"]]X = train[features]y = train[target]X_test = test[features]models = [    LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=63, subsample=0.8, colsample_bytree=0.8, random_state=42),    LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.7, colsample_bytree=0.7, random_state=2024),    LGBMClassifier(n_estimators=2000, learning_rate=0.02, num_leaves=95, subsample=0.9, colsample_bytree=0.9, random_state=777)]skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)oof = np.zeros(len(X))test_preds = np.zeros(len(X_test))for model in models:    fold_preds = np.zeros(len(X_test))    for tr_idx, val_idx in skf.split(X, y):        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc", verbose=100)        oof[val_idx] += model.predict_proba(X_val)[:,1] / len(models)        fold_preds += model.predict_proba(X_test)[:,1] / skf.n_splits    test_preds += fold_preds / len(models)print("OOF:", roc_auc_score(y, oof))rank_preds = pd.Series(test_preds).rank() / len(test_preds)final_preds = (test_preds + rank_preds) / 2submission = pd.DataFrame({    "ID": test["ID"],    "probability": final_preds})submission.to_csv("v15_full_submission.csv", index=False)print("done")